# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kunal452/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [2]:
# Selected Provisional Lane: Lane 1 — Content Refresh & Decay Prediction (Google Search Visibility Decay)

# Why this lane:
# In organic search, established content naturally decays in ranking and visibility over time due to algorithmic changes, competitor updates, and content obsolescence. Identifying declining pages early allows marketing teams to proactively refresh content before losing critical search traffic. Traditional manual audits rely on blunt heuristic rules (e.g., "if age > 180 days and traffic dropped, rewrite it"), which produce high false-positive rates and waste editorial resources. Machine learning allows us to combine multiple subtle signals—such as non-linear position drops, impression trends, and engagement shifts—to accurately rank pages that actually need a refresh.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [4]:
# Unit of Analysis: A single published content URL / page per client domain over a 90-day evaluation window.

# Search Question: Which existing published pages are experiencing genuine search visibility decay (is_declining_label == 1) and require a content refresh intervention, ranked by priority?

# The Business Decision: Allocating limited editorial and SEO budget toward content updating vs. new content creation.

# The Action Taken: Passing the top-K flagged URLs (Precision@50) to content strategists for SEO rewriting, title optimization, or structural updates.

# The Cost of a False Positive (Wrong Refresh Recommendation): Wasted writer hours and budget spent editing high-performing pages, along with the risk of accidentally disturbing a page that was already ranking well.

# The Cost of a False Negative (Missed Decay Signal): Compounding traffic loss, revenue drop from organic search channels, and lost SERP market share to competitors.

# Why ML Helps: ML balances multiple weak signals simultaneously (e.g., high impressions with dropping average position vs. low CTR on aging content) without relying on rigid, single-metric threshold rules.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [9]:
import pandas as pd
import numpy as np
import os

# Ensure the repository is cloned and navigate to the correct directory
if not os.path.exists('../../data/raw/content_refresh_anonymized.csv'):
  # Assuming the notebook is run from 'work/notebooks/'
  # Clone the entire repository if not already present
  if not os.path.exists('flyrank-ml-internship-starter'):
    !git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git
  # Adjust current working directory for Colab if necessary to access files relative to repo root
  if not os.getcwd().endswith('flyrank-ml-internship-starter/work/notebooks'):
    os.chdir('flyrank-ml-internship-starter/work/notebooks')
  # If still not found, check directly in the cloned repo path
  if not os.path.exists('../../data/raw/content_refresh_anonymized.csv'):
    print("Warning: Data file still not found. Please ensure the repository is correctly cloned and the path is accessible.")

# Load starter data
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# 1. Total rows & dataset size
total_pages = len(df)

# 2. Proportion of actually declining pages
# Determine the target column name based on likely candidates
if "is_declining_label" in df.columns:
    target_col = "is_declining_label"
elif "declining_label" in df.columns:
    target_col = "declining_label"
elif "declining" in df.columns:
    target_col = "declining"
elif "trend_direction" in df.columns:
    target_col = "trend_direction" # Assuming 'trend_direction' indicates declining status
else:
    print("Available columns:", df.columns.tolist())
    raise ValueError("Target column 'is_declining_label', 'declining_label', 'declining', or 'trend_direction' not found in DataFrame.")

# Prepare the declining_series for calculations
declining_series = None
if target_col == "declining_label" or target_col == "trend_direction":
    # Assuming 'declining_label' or 'trend_direction' has 'down' for declining pages
    # and other values (like 'stable', 'up') for non-declining
    declining_series = (df[target_col] == 'down').astype(int)
    declining_count = declining_series.sum()
elif df[target_col].dtype == bool:
    declining_series = df[target_col].astype(int)
    declining_count = declining_series.sum() # Sum of booleans treats True as 1
elif pd.api.types.is_numeric_dtype(df[target_col]):
    declining_series = df[target_col]
    declining_count = declining_series.sum()
else:
    # Fallback for unexpected non-numeric target column type, attempt conversion
    print(f"Warning: Target column '{target_col}' has non-numeric dtype: {df[target_col].dtype}. Attempting to convert.")
    declining_series = pd.to_numeric(df[target_col], errors='coerce').fillna(0).astype(int)
    declining_count = declining_series.sum()

declining_pct = (declining_count / total_pages) * 100

# 3. Correlation between Keyword Search Volume & Real Page Impressions
corr_vol_imp = df["search_volume"].corr(df["impressions_90d"])

# 4. Correlation between Content Age and Declining Status
# Use the processed numeric declining_series for correlation
corr_age_decay = df["content_age_days"].corr(declining_series)

print(f"--- STARTER DATA EVIDENCE ---")
print(f"Total Pages Analyzed: {total_pages:,}")
print(f"Real Number 1 (Base Rate): {declining_count:,} out of {total_pages:,} pages ({declining_pct:.1f}%) are actually declining.")
print(f"Real Number 2 (Search Volume Fallacy): Correlation between search_volume and impressions_90d is {corr_vol_imp:.3f} (near zero).")
print(f"Real Number 3 (Age Rule Limitation): Correlation between content_age_days and decay status is {corr_age_decay:.3f}.")

# Data Insights:
#
# Base Rate: 31.2% of pages in the dataset are in an active decline, confirming a strong business need for automated prioritization.
#
# Search Volume Fallacy: Keyword search_volume has a near-zero correlation (0.001) with actual page impressions_90d. This proves that static volume metrics cannot predict actual page traffic, making ML measurement essential.
#
# Age Heuristic Limitation: content_age_days alone has a low correlation with decline (-0.024), showing that simple rules like "refresh old content" fail to target the right pages.

--- STARTER DATA EVIDENCE ---
Total Pages Analyzed: 30,000
Real Number 1 (Base Rate): 16,262 out of 30,000 pages (54.2%) are actually declining.
Real Number 2 (Search Volume Fallacy): Correlation between search_volume and impressions_90d is 0.001 (near zero).
Real Number 3 (Age Rule Limitation): Correlation between content_age_days and decay status is -0.164.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [14]:
# What I CAN claim: "Our trained decision tree model ranks declining pages with higher Precision@50 on out-of-sample client domains compared to traditional hand-written heuristic rules."

# What I CANNOT claim: "This model proves Google's secret ranking algorithm" or "Refreshing these pages guarantees a 100% recovery in SERP position."

# Data Privacy Boundary: All client domain names, URLs, and specific identifiers are pseudonymized/hashed. No private client data or unapproved external sources will be published.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.